# 🎯 RS-LiDAR & LiDAR: Chấm Lại Điểm GenEval Chuẩn Bài Báo (Mask2Former Swin-S)
### Đánh giá tự động 1-Click (Run All) cho cả 3 Settings:
1. **SD v1.5 (DDIM-50, $\eta=0.0$)**
2. **SD v1.5 (DDPM-100, $\eta=1.0$)**
3. **SDXL 2.6B (DDPM-100, $\eta=1.0$)**

---
### 📌 Hướng Dẫn Sử Dụng (Chỉ Cần Bấm Run All):
- Notebook này chạy **trên tài khoản Google Drive cá nhân** của bạn.
- **Chỉ cần bấm**: `Runtime` -> `Run all` (hoặc `Ctrl + F9`).
- Notebook sẽ:
  1. Gắn kết Google Drive (`/content/drive/MyDrive/RS-LiDAR`).
  2. Tự động nạp mô hình chuẩn **Mask2Former Swin-S COCO** (`facebook/mask2former-swin-small-coco-instance`) và **CLIP-ViT-B/32**.
  3. Tự động quét toàn bộ các thư mục thực nghiệm trong `Target_samples/` của cả 3 settings.
  4. Chấm điểm chi tiết 553 prompt cho từng thí nghiệm và lưu `geneval_summary.csv` trực tiếp vào từng folder trên Drive.
  5. Xuất **Bảng Tổng Hợp Master Khoa Học** so sánh toàn diện giữa Vanilla LiDAR, RS-LiDAR và Bài Báo Bảng 2 trên cả 3 settings!


## 1. Cài Đặt Môi Trường & Kiểm Tra GPU

In [ ]:
# @title 🚀 Cài đặt thư viện đánh giá chuẩn khoa học
!nvidia-smi

import os, sys, glob, json, time, shutil
import numpy as np
import pandas as pd
from PIL import Image
from tqdm.auto import tqdm

# Cài đặt transformers, timm và các phụ thuộc cần thiết cho Mask2Former Native
!pip install -q --upgrade transformers timm tabulate

import torch
from transformers import AutoImageProcessor, Mask2FormerForUniversalSegmentation, CLIPProcessor, CLIPModel

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"\n✅ Môi trường PyTorch sẵn sàng! Sử dụng thiết bị: {device}")
if device == "cuda":
    print(f"   GPU: {torch.cuda.get_device_name(0)}")
    print(f"   VRAM: {torch.cuda.get_device_properties(0).total_memory / (1024**3):.2f} GB")


## 2. Gắn Kết Google Drive Cá Nhân & Xác Định Thư Mục Dữ Liệu

In [ ]:
# @title 📁 Gắn kết Google Drive cá nhân
from google.colab import drive
drive.mount('/content/drive')

base_drive = '/content/drive/My Drive' if os.path.exists('/content/drive/My Drive') else '/content/drive/MyDrive'
assert os.path.exists(base_drive), f"❌ Không tìm thấy Google Drive tại {base_drive}!"

# Tìm thư mục RS-LiDAR trên Drive cá nhân
candidate_dirs = [
    f"{base_drive}/RS-LiDAR",
    f"{base_drive}/RS-LiDAR-Personal",
    f"{base_drive}/RS-LiDAR (1)"
]

DRIVE_DIR = None
for c in candidate_dirs:
    if os.path.exists(c):
        DRIVE_DIR = c
        break

if DRIVE_DIR is None:
    DRIVE_DIR = f"{base_drive}/RS-LiDAR"

TARGET_BASE = f"{DRIVE_DIR}/Target_samples"
assert os.path.exists(TARGET_BASE), f"❌ Không tìm thấy thư mục Target_samples tại: {TARGET_BASE}! Hãy kiểm tra lại dữ liệu trên Drive."

print(f"✅ Đã kết nối thành công với Google Drive cá nhân!")
print(f"📁 Thư mục gốc RS-LiDAR: {DRIVE_DIR}")
print(f"🎯 Thư mục Target_samples: {TARGET_BASE}")


## 3. Tải Metadata GenEval & Khởi Tạo Mask2Former Swin-S Chuẩn Bài Báo

In [ ]:
# @title 📦 Nạp mô hình Mask2Former Swin-S & CLIP (Chuẩn GenEval gốc)

# 1. Tải metadata 553 prompts chính thức của GenEval
meta_url = "https://raw.githubusercontent.com/leekwanreal/RS-LiDAR/main/prompt_files/geneval_metadata.jsonl"
meta_path = "geneval_metadata.jsonl"
if not os.path.exists(meta_path):
    print("⬇️ Đang tải metadata GenEval...")
    !wget -q {meta_url} -O {meta_path}

prompts_meta = []
with open(meta_path, 'r', encoding='utf-8') as f:
    for line in f:
        if line.strip():
            prompts_meta.append(json.loads(line))
print(f"✅ Đã tải thành công metadata cho {len(prompts_meta)} prompts GenEval.")

# 2. Nạp mô hình Mask2Former Swin-S COCO
print("\n📦 Đang nạp mô hình Mask2Former Swin-S COCO (facebook/mask2former-swin-small-coco-instance)...\n   (Chính xác mô hình chuẩn của bài báo GenEval, mAP ~52%)")
mask2former_id = "facebook/mask2former-swin-small-coco-instance"
image_processor = AutoImageProcessor.from_pretrained(mask2former_id)
detector = Mask2FormerForUniversalSegmentation.from_pretrained(mask2former_id).to(device).eval()
id2label = detector.config.id2label

# 3. Nạp mô hình CLIP ViT-B/32 để phân loại màu sắc chính xác
print("📦 Đang nạp mô hình CLIP ViT-B/32 cho kiểm tra thuộc tính màu sắc (color_attr)...\n")
clip_model = CLIPModel.from_pretrained("openai/clip-vit-base-patch32").to(device).eval()
clip_processor = CLIPProcessor.from_pretrained("openai/clip-vit-base-patch32")

COLORS = ['red', 'orange', 'yellow', 'green', 'blue', 'purple', 'pink', 'brown', 'black', 'white']
color_prompts = [f"a photo of a {c} object" for c in COLORS]

def classify_crop_color(crop_img):
    inputs = clip_processor(text=color_prompts, images=crop_img, return_tensors="pt", padding=True).to(device)
    with torch.inference_mode():
        outputs = clip_model(**inputs)
        best_idx = outputs.logits_per_image.argmax(dim=-1).item()
        return COLORS[best_idx]

print("🎉 Toàn bộ hệ thống đánh giá GenEval chuẩn đã sẵn sàng!")


## 4. Quét Tự Động Toàn Bộ Thư Mục Thực Nghiệm Của Cả 3 Settings

In [ ]:
# @title 🔍 Quét và phân loại thư mục thực nghiệm

# Tìm tất cả thư mục con trong Target_samples có chứa ảnh prompt
all_subdirs = sorted(glob.glob(f"{TARGET_BASE}/*"))
valid_experiments = []

for d in all_subdirs:
    if not os.path.isdir(d):
        continue
    prompt_dirs = glob.glob(f"{d}/[0-9]*")
    if len(prompt_dirs) >= 10:  # Thư mục có ít nhất 10 prompts đã sinh ảnh
        d_name = os.path.basename(d)
        
        # Phân loại setting
        setting_label = "Khác"
        if "SDXL" in d_name:
            setting_label = "SDXL DDPM-100"
        elif "DDIM" in d_name or "Step50" in d_name:
            setting_label = "SD 1.5 DDIM-50"
        elif "DDPM" in d_name or "Step100" in d_name:
            setting_label = "SD 1.5 DDPM-100"
            
        is_rs = "RSLiDAR" in d_name or "sig" in d_name or "smoothing" in d_name.lower()
        method_label = "RS-LiDAR (Đề xuất)" if is_rs else "Vanilla LiDAR (Đối chứng)"
        
        valid_experiments.append({
            "path": d,
            "name": d_name,
            "setting": setting_label,
            "method": method_label,
            "prompt_count": len(prompt_dirs)
        })

print(f"🎯 ĐÃ TÌM THẤY {len(valid_experiments)} THỰC NGHIỆM TRÊN GOOGLE DRIVE CẦN CHẤM ĐIỂM:")
print("-" * 85)
for i, exp in enumerate(valid_experiments, 1):
    print(f"{i:2d}. [{exp['setting']}] {exp['method']}: {exp['name']} ({exp['prompt_count']}/553 prompts)")
print("-" * 85)


## 5. Chạy Chấm Điểm GenEval Tự Động (Mask2Former Swin-S)

In [ ]:
# @title 🚀 BẮT ĐẦU CHẤM ĐIỂM TỰ ĐỘNG CHO TOÀN BỘ CÁC SETTING

def evaluate_geneval_for_folder(target_dir, exp_name):
    """Đánh giá toàn bộ ảnh trong target_dir bằng Mask2Former Swin-S và lưu geneval_summary.csv"""
    # 1. Quét ảnh trong các thư mục prompt
    prompt_dir_map = {}
    for p_dir in glob.glob(f"{target_dir}/[0-9]*"):
        try:
            p_idx = int(os.path.basename(p_dir))
        except ValueError:
            continue
        imgs = sorted(glob.glob(f"{p_dir}/samples/*.png")) or [f for f in glob.glob(f"{p_dir}/*.png") if not f.endswith("grid.png")]
        if imgs:
            prompt_dir_map[p_idx] = imgs

    if not prompt_dir_map:
        print(f"⚠️ Không tìm thấy ảnh hợp lệ trong: {exp_name}")
        return None

    print(f"\n" + "="*75)
    print(f"🎯 Đang chấm GenEval cho: {exp_name}")
    print(f"   Số prompt có ảnh: {len(prompt_dir_map)}/553")
    print("="*75)

    task_results = {'single_object': [], 'two_object': [], 'counting': [], 'colors': [], 'position': [], 'color_attr': []}
    
    for p_idx in tqdm(sorted(prompt_dir_map.keys()), desc=f"Chấm điểm: {exp_name[:30]}..."):
        if p_idx >= len(prompts_meta):
            continue
        meta = prompts_meta[p_idx]
        tag = meta.get('tag', 'single_object')
        if tag not in task_results:
            continue

        imgs = prompt_dir_map[p_idx]
        prompt_scores = []
        for img_path in imgs:
            try:
                img = Image.open(img_path).convert('RGB')
                inputs = image_processor(images=img, return_tensors="pt").to(device)
                with torch.inference_mode():
                    outputs = detector(**inputs)

                # Instance Segmentation threshold = 0.5 chuẩn Mask2Former
                results = image_processor.post_process_instance_segmentation(
                    outputs, target_sizes=[img.size[::-1]], threshold=0.5
                )[0]
                segmentation = results["segmentation"].detach().cpu().numpy()
                segments_info = results["segments_info"]

                detected_objects = []
                for seg in segments_info:
                    c_name = id2label[seg["label_id"]].lower()
                    mask = (segmentation == seg["id"])
                    y_indices, x_indices = np.where(mask)
                    if len(x_indices) > 0 and len(y_indices) > 0:
                        x1, x2 = float(np.min(x_indices)), float(np.max(x_indices))
                        y1, y2 = float(np.min(y_indices)), float(np.max(y_indices))
                        box = [x1, y1, x2, y2]
                        center_x = (x1 + x2) / 2.0
                        center_y = (y1 + y2) / 2.0

                        if (x2 - x1 > 12 and y2 - y1 > 12):
                            crop = img.crop((max(0, x1), max(0, y1), min(img.width, x2), min(img.height, y2)))
                            pred_color = classify_crop_color(crop)
                        else:
                            pred_color = 'unknown'

                        detected_objects.append({
                            'class': c_name,
                            'box': box,
                            'center_x': center_x,
                            'center_y': center_y,
                            'color': pred_color
                        })

                # Đánh giá theo rule bài báo GenEval
                success = False
                includes = meta.get('include', [])
                if tag == 'single_object':
                    req_cls = includes[0]['class'].lower()
                    success = any(req_cls in obj['class'] or obj['class'] in req_cls for obj in detected_objects)
                elif tag == 'two_object':
                    req1, req2 = includes[0]['class'].lower(), includes[1]['class'].lower()
                    success = any(req1 in obj['class'] or obj['class'] in req1 for obj in detected_objects) and \
                               any(req2 in obj['class'] or obj['class'] in req2 for obj in detected_objects)
                elif tag == 'counting':
                    req_cls, target_count = includes[0]['class'].lower(), includes[0]['count']
                    found_count = sum(1 for obj in detected_objects if req_cls in obj['class'] or obj['class'] in req_cls)
                    success = (found_count == target_count)
                elif tag == 'colors':
                    req_cls, req_color = includes[0]['class'].lower(), includes[0]['color'].lower()
                    success = any((req_cls in obj['class'] or obj['class'] in req_cls) and (obj['color'] == req_color) for obj in detected_objects)
                elif tag == 'position':
                    req1, req2 = includes[0]['class'].lower(), includes[1]['class'].lower()
                    pos_type = includes[1].get('position', ['right of', 0])[0]
                    o1_list = [o for o in detected_objects if req1 in o['class'] or o['class'] in req1]
                    o2_list = [o for o in detected_objects if req2 in o['class'] or o['class'] in req2]
                    if o1_list and o2_list:
                        o1, o2 = o1_list[0], o2_list[0]
                        if 'right' in pos_type: success = (o2['center_x'] > o1['center_x'])
                        elif 'left' in pos_type: success = (o2['center_x'] < o1['center_x'])
                        elif 'above' in pos_type or 'top' in pos_type: success = (o2['center_y'] < o1['center_y'])
                        elif 'below' in pos_type or 'bottom' in pos_type: success = (o2['center_y'] > o1['center_y'])
                        else: success = True
                elif tag == 'color_attr':
                    success = all(any((inc['class'].lower() in obj['class'] or obj['class'] in inc['class'].lower()) and (obj['color'] == inc['color'].lower()) for obj in detected_objects) for inc in includes)

                prompt_scores.append(1.0 if success else 0.0)
            except Exception as e:
                pass

        if prompt_scores:
            task_results[tag].append(np.mean(prompt_scores))

    # Tổng hợp kết quả từng task
    summary_rows = []
    all_means = []
    for t_name, scores in task_results.items():
        mean_val = np.mean(scores) if scores else 0.0
        if scores: all_means.append(mean_val)
        summary_rows.append({'Nhiệm Vụ (Task)': t_name, 'Số Lượng Prompt': len(scores), 'Độ Chính Xác (Accuracy ↑)': f"{mean_val:.4f}"})

    overall_geneval = float(np.mean(all_means)) if all_means else 0.0
    summary_rows.append({'Nhiệm Vụ (Task)': '🔥 OVERALL GENEVAL BENCHMARK', 'Số Lượng Prompt': sum(len(s) for s in task_results.values()), 'Độ Chính Xác (Accuracy ↑)': f"{overall_geneval:.4f}"})
    
    df_geneval = pd.DataFrame(summary_rows)
    csv_path = f"{target_dir}/geneval_summary.csv"
    df_geneval.to_csv(csv_path, index=False)
    print(f"💾 Đã lưu bảng điểm GenEval chuẩn Mask2Former tại: {csv_path}")
    print(f"⭐ ĐIỂM OVERALL GENEVAL: {overall_geneval:.4f}")
    return overall_geneval

# Chạy vòng lặp cho tất cả thực nghiệm
rescore_results = {}
start_total = time.time()

for exp in valid_experiments:
    score = evaluate_geneval_for_folder(exp["path"], exp["name"])
    if score is not None:
        rescore_results[exp["name"]] = score

elapsed_total = time.time() - start_total
print("\n" + "="*75)
print(f"🎉 HOÀN THÀNH CHẤM ĐIỂM TẤT CẢ THỰC NGHIỆM! (Tổng thời gian: {elapsed_total/60:.1f} phút)")
print("="*75)


## 6. Xuất Bảng Tổng Hợp Master Khoa Học Cho Cả 3 Settings

In [ ]:
# @title 📊 TỔNG HỢP KẾT QUẢ KHOA HỌC MASTER (3 SETTINGS)

def get_metrics_from_folder(folder_path):
    """Đọc các chỉ số ImageReward, CLIP, HPS v2.1 và GenEval từ folder"""
    # 1. Đọc GenEval từ geneval_summary.csv
    ge_score = "N/A"
    ge_csv = f"{folder_path}/geneval_summary.csv"
    if os.path.exists(ge_csv):
        try:
            df_ge = pd.read_csv(ge_csv)
            for _, r in df_ge.iterrows():
                t_name = str(r.iloc[0]).upper()
                if 'OVERALL' in t_name or 'GENEVAL' in t_name:
                    ge_score = f"{float(r.iloc[2]):.4f}"
                    break
        except Exception:
            pass

    # 2. Đọc IR, CLIP, HPS từ final_metrics.json hoặc results.json
    fm_json = f"{folder_path}/final_metrics.json"
    ir_val, clip_val, hps_val, count = None, None, None, 0
    if os.path.exists(fm_json):
        try:
            with open(fm_json, 'r') as f:
                d = json.load(f)
                ir_val = d.get('ImageReward', d.get('ir'))
                clip_val = d.get('CLIP', d.get('clip'))
                hps_val = d.get('HPS', d.get('hps'))
                count = d.get('total_prompts', 553)
        except Exception:
            pass

    # Fallback quét results.json
    if ir_val is None:
        r_files = glob.glob(f"{folder_path}/[0-9]*/results.json")
        if r_files:
            irs, clips, hpss = [], [], []
            for rf in r_files:
                try:
                    with open(rf, 'r') as f:
                        d = json.load(f)
                    if 'image_reward' in d: irs.append(d['image_reward'])
                    if 'clip_score' in d: clips.append(d['clip_score'])
                    if 'hps_score' in d: hpss.append(d['hps_score'])
                except Exception:
                    pass
            if irs: ir_val = np.mean(irs)
            if clips: clip_val = np.mean(clips)
            if hpss: hps_val = np.mean(hpss)
            count = len(r_files)

    return {
        "ir": f"{ir_val:.4f}" if ir_val is not None else "N/A",
        "clip": f"{clip_val:.4f}" if clip_val is not None else "N/A",
        "hps": f"{hps_val:.4f}" if hps_val is not None else "N/A",
        "ge": ge_score,
        "count": count
    }

# Xây dựng bảng so sánh Master theo đúng tiêu chuẩn công bố khoa học
master_rows = [
    # --- SETTING 1: SD 1.5 DDIM-50 ---
    {"Setting": "SD 1.5 DDIM-50", "Phương Pháp": "SD v1.5 Gốc (Chưa lái)", "ImageReward ↑": "-0.125", "CLIP-Score ↑": "0.269", "HPS v2.1 ↑": "0.270", "GenEval ↑": "0.423", "Nguồn Dữ Liệu": "Bài Báo Bảng 2"},
    {"Setting": "SD 1.5 DDIM-50", "Phương Pháp": "DATE (Na et al., 2025)", "ImageReward ↑": "0.097", "CLIP-Score ↑": "0.271", "HPS v2.1 ↑": "0.261", "GenEval ↑": "0.419", "Nguồn Dữ Liệu": "Bài Báo Bảng 2"},
    {"Setting": "SD 1.5 DDIM-50", "Phương Pháp": "LiDAR Bảng 2 (Công bố)", "ImageReward ↑": "0.378", "CLIP-Score ↑": "0.278", "HPS v2.1 ↑": "0.277", "GenEval ↑": "0.475", "Nguồn Dữ Liệu": "Bài Báo Bảng 2"},
]

# Thêm dữ liệu thực nghiệm SD 1.5 DDIM
for exp in valid_experiments:
    if exp["setting"] == "SD 1.5 DDIM-50":
        m = get_metrics_from_folder(exp["path"])
        master_rows.append({
            "Setting": "SD 1.5 DDIM-50",
            "Phương Pháp": f"🔥 {exp['method']}",
            "ImageReward ↑": m["ir"],
            "CLIP-Score ↑": m["clip"],
            "HPS v2.1 ↑": m["hps"],
            "GenEval ↑": m["ge"],
            "Nguồn Dữ Liệu": f"Thực nghiệm (n={m['count']})"
        })

# --- SETTING 2: SD 1.5 DDPM-100 ---
master_rows.extend([
    {"Setting": "SD 1.5 DDPM-100", "Phương Pháp": "SD v1.5 Gốc (Chưa lái)", "ImageReward ↑": "-0.001", "CLIP-Score ↑": "0.271", "HPS v2.1 ↑": "0.263", "GenEval ↑": "0.426", "Nguồn Dữ Liệu": "Bài Báo Bảng 2"},
    {"Setting": "SD 1.5 DDPM-100", "Phương Pháp": "DATE (Na et al., 2025)", "ImageReward ↑": "0.364", "CLIP-Score ↑": "0.274", "HPS v2.1 ↑": "0.267", "GenEval ↑": "0.438", "Nguồn Dữ Liệu": "Bài Báo Bảng 2"},
    {"Setting": "SD 1.5 DDPM-100", "Phương Pháp": "LiDAR Bảng 2 (Công bố)", "ImageReward ↑": "0.384", "CLIP-Score ↑": "0.278", "HPS v2.1 ↑": "0.276", "GenEval ↑": "0.478", "Nguồn Dữ Liệu": "Bài Báo Bảng 2"},
])

# Thêm dữ liệu thực nghiệm SD 1.5 DDPM
for exp in valid_experiments:
    if exp["setting"] == "SD 1.5 DDPM-100":
        m = get_metrics_from_folder(exp["path"])
        master_rows.append({
            "Setting": "SD 1.5 DDPM-100",
            "Phương Pháp": f"🔥 {exp['method']}",
            "ImageReward ↑": m["ir"],
            "CLIP-Score ↑": m["clip"],
            "HPS v2.1 ↑": m["hps"],
            "GenEval ↑": m["ge"],
            "Nguồn Dữ Liệu": f"Thực nghiệm (n={m['count']})"
        })

# --- SETTING 3: SDXL DDPM-100 ---
master_rows.extend([
    {"Setting": "SDXL DDPM-100", "Phương Pháp": "SDXL Base (Chưa lái)", "ImageReward ↑": "0.722", "CLIP-Score ↑": "0.282", "HPS v2.1 ↑": "0.292", "GenEval ↑": "0.545", "Nguồn Dữ Liệu": "Bài Báo Bảng 2"},
    {"Setting": "SDXL DDPM-100", "Phương Pháp": "DATE (Na et al., 2025)", "ImageReward ↑": "0.960", "CLIP-Score ↑": "0.283", "HPS v2.1 ↑": "0.294", "GenEval ↑": "0.570", "Nguồn Dữ Liệu": "Bài Báo Bảng 2"},
    {"Setting": "SDXL DDPM-100", "Phương Pháp": "LiDAR Bảng 2 (Công bố)", "ImageReward ↑": "1.006", "CLIP-Score ↑": "0.285", "HPS v2.1 ↑": "0.302", "GenEval ↑": "0.598", "Nguồn Dữ Liệu": "Bài Báo Bảng 2"},
])

# Thêm dữ liệu thực nghiệm SDXL DDPM
for exp in valid_experiments:
    if exp["setting"] == "SDXL DDPM-100":
        m = get_metrics_from_folder(exp["path"])
        master_rows.append({
            "Setting": "SDXL DDPM-100",
            "Phương Pháp": f"🔥 {exp['method']}",
            "ImageReward ↑": m["ir"],
            "CLIP-Score ↑": m["clip"],
            "HPS v2.1 ↑": m["hps"],
            "GenEval ↑": m["ge"],
            "Nguồn Dữ Liệu": f"Thực nghiệm (n={m['count']})"
        })

master_df = pd.DataFrame(master_rows)
print("\n" + "="*95)
print("📊 BẢNG TỔNG HỢP MASTER KHOA HỌC CẢ 3 SETTINGS (MASK2FORMER GENEVAL CHUẨN)")
print("="*95)
display(master_df)

# Lưu ra Google Drive
master_csv_path = f"{TARGET_BASE}/geneval_master_summary_3settings.csv"
master_df.to_csv(master_csv_path, index=False)
print("\n" + "="*95)
print(f"💾 Đã lưu bảng tổng hợp Master ra Google Drive tại:\n   {master_csv_path}")
print("="*95)
